# SRA“损伤”（突触破坏）实验

在这个演示中，我们进行了一个有点像黑客的实验，以证明 SRA 训练的网络是**“按功能完全模块化”**。

1. 在`copy`和`reverse`任务上同时训练模型。
2. **故意破坏（设置为零）`reverse` 任务中经常使用的特定专家（突触）**的权重。
3. 确认销毁后，`reverse` 任务无法再求解，但**`copy` 任务完全不受影响（保持100% 准确率）**。

## 1. 环境设置和模型准备

In [1]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
torch.manual_seed(42)
import torch.nn.functional as F
torch.manual_seed(42)
from src.sra_gpu_models import MoESRAModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

config = MoESRAConfig(
    vocab_size=30,
    d_model=64,
    n_layers=2,
    n_heads=2,
    num_synapses=8,
    k=2,
    max_seq_len=32
)
model = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=128).to(device)
optimizer = make_optimizer(model, lr=0.005)

Using device: cpu


## 2. 执行多任务学习

In [2]:
print("Training started... (Copy & Reverse)")
model.train()
epochs = 1000
batch_size = 32
tasks = ["copy", "reverse"]

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("Training finished!")

Training started... (Copy & Reverse)


Epoch 100/1000 | Loss: 1.3255


Epoch 200/1000 | Loss: 0.7296


Epoch 300/1000 | Loss: 0.3903


Epoch 400/1000 | Loss: 0.5676


...

Epoch 600/1000 | Loss: 0.3406


Epoch 700/1000 | Loss: 0.2710


Epoch 800/1000 | Loss: 0.4116


Epoch 900/1000 | Loss: 0.3624


Epoch 1000/1000 | Loss: 0.4443
Training finished!


## 3. 损伤前性能测试和目标突触识别
确认两个任务都可以解决，并找到`reverse`任务（目标）中最常用的突触。

In [3]:
def test_task(task_name):
    model.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model(x, y_in)
        preds = outputs.argmax(dim=-1)
        
    # Percentage of correct answers excluding padding
    mask = y != 0 # 0 is PAD
    acc = (preds[mask] == y[mask]).float().mean().item()
    
    # Aggregate used synapses (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)
    
    print(f"[{task_name.upper()}] Accuracy: {acc*100:.1f}%")
    return usage

print("=== Before Lesion ===")
copy_usage = test_task("copy")
reverse_usage = test_task("reverse")

# Target synapses that are more active in reverse than in copy
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]
print(f"\n=> Target for destruction: Synapses {target_synapses} (highly active in REVERSE)")
# Set the synapse most used in Reverse task as 'destruction target'
target_synapses = [i for i, val in enumerate(diff) if val > 0]
print(f"\n=> Target for destruction: Synapse {target_synapses} (highly active in REVERSE)")

=== Before Lesion ===
[COPY] Accuracy: 99.4%
[REVERSE] Accuracy: 67.4%

=> Target for destruction: Synapses [6] (highly active in REVERSE)

=> Target for destruction: Synapse [6] (highly active in REVERSE)


## 4. 突触破坏（病变）
故意用零覆盖目标突触的权重。

In [4]:
print(f"Destroying Synapses {target_synapses} in all layers...")

with torch.no_grad():
    # Set specific expert weights in MoESRABlock to 0
    for layer_to_destroy in range(config.n_layers):
        block = model.blocks[layer_to_destroy]
        for target_synapse in target_synapses:
            block.w1.data[target_synapse].zero_()
            block.b1.data[target_synapse].zero_()
            block.w2.data[target_synapse].zero_()
            block.b2.data[target_synapse].zero_()

print("Destruction complete. 💥")

Destroying Synapses [6] in all layers...
Destruction complete. 💥


## 5. 损伤后性能测试
重新测试大脑的一部分（突触）被破坏。
`reverse` 的准确度会下降，但`copy`应该使用不同的突触，但不会受到影响！

In [5]:
print("=== After Lesion ===")
test_task("copy")
test_task("reverse")

print("\n💡 [Insight]")
print("If you destroy a part of a large monolithic neural network (like a standard Transformer) with zeros, usually all tasks are devastated.")
print("However, in SRA, because independent expert pathways (synapses) are formed for each task,")
print("it demonstrates astonishing robustness: 'Even if the brain for Reverse breaks, the brain for Copy continues to work flawlessly'!")

=== After Lesion ===
[COPY] Accuracy: 94.6%
[REVERSE] Accuracy: 60.9%

💡 [Insight]
If you destroy a part of a large monolithic neural network (like a standard Transformer) with zeros, usually all tasks are devastated.
However, in SRA, because independent expert pathways (synapses) are formed for each task,
it demonstrates astonishing robustness: 'Even if the brain for Reverse breaks, the brain for Copy continues to work flawlessly'!
